In [2]:
import pandas as pd
import numpy as np
import scanpy as sc

from sklearn.metrics import roc_auc_score

In [ ]:
#adata = sc.read('validation/diff_gene/stagei/Lung_Cancer_HD_Only_Experiment1Macrophage.h5ad')

In [51]:
adata1 = sc.read('/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/VisiumHD/processed/Lung_Cancer_HD_Only_Experiment1Endothelial.h5ad')
adata2 = sc.read('/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/VisiumHD/processed/Breast_Cancer_FreshEndothelial.h5ad')
adata3 = sc.read('/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/VisiumHD/processed/Colon_Cancer_P1Endothelial.h5ad')
adata4 = sc.read('/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/VisiumHD/processed/Colon_Cancer_P2Endothelial.h5ad')
adata5 = sc.read('/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/VisiumHD/processed/Colon_Cancer_P5Endothelial.h5ad')
adata6 = sc.read('/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/VisiumHD/processed/HumanLungCancerEndothelial.h5ad')
adata7 = sc.read('/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/VisiumHD/processed/Lung_Cancer_Fixed_FrozenEndothelial.h5ad')

adatas = [adata1,adata2, adata3,adata4, adata5, adata6, adata7]
adata = sc.concat(adatas, join='inner', axis=0) 

/work/DPDS/s439765/envs/spatial_tcr/lib/python3.9/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [52]:
adata.obs.columns

Index(['in_tissue', 'array_row', 'array_col', 'X', 'Y', 'n_genes',
       'tumor_gene_signature', 'stromal_immune_gene_signature',
       't_gene_signature', 'b_gene_signature', 'neutrophil_gene_signature',
       'macrophage_gene_signature', 'endothelial_gene_signature',
       'fibroblast_gene_signature', 'leiden', 'T', 'B', 'Endothelial',
       'cell_type'],
      dtype='object')

In [53]:
subset = adata[adata.obs['cell_type'] == 1]
subset.obs

,in_tissue,array_row,array_col,X,Y,n_genes,tumor_gene_signature,stromal_immune_gene_signature,t_gene_signature,b_gene_signature,neutrophil_gene_signature,macrophage_gene_signature,endothelial_gene_signature,fibroblast_gene_signature,leiden,T,B,Endothelial,cell_type
s_008um_00123_00164-1,1,123,164,1312,984,288,6.869911,17.174776,0.000000,0.000000,0.000000,0.000000,3.434955,4.111858,1,0,0,0,1
s_008um_00013_00648-1,1,13,648,5184,104,275,11.581820,24.508215,0.000000,0.000000,3.501174,3.501174,0.000000,0.000000,0,0,0,0,1
s_008um_00161_00253-1,1,161,253,2024,1288,361,7.095243,32.688744,0.000000,3.211228,0.000000,0.000000,0.000000,3.884016,1,0,0,0,1
s_008um_00401_00407-1,1,401,407,3256,3208,386,3.168805,31.331490,0.000000,7.407785,3.168805,0.000000,0.000000,0.000000,2,0,0,0,1
s_008um_00286_00660-1,1,286,660,5280,2288,557,10.020837,19.375395,2.767914,0.000000,2.767914,0.000000,0.000000,0.000000,2,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
s_008um_00267_00390-1,1,267,390,3120,2136,751,8.980241,99.551697,2.431281,5.510752,7.942033,5.900770,3.079471,6.399065,11,0,0,0,1
s_008um_00224_00258-1,1,224,258,2064,1792,248,0.000000,56.417103,0.000000,7.085223,0.000000,7.085223,0.000000,8.164355,4,0,0,0,1
s_008um_00016_00312-1,1,16,312,2496,128,368,0.000000,35.854542,0.000000,3.168805,0.000000,0.000000,0.000000,4.523050,4,0,0,0,1
s_008um_00339_00720-1,1,339,720,5760,2712,269,3.485709,48.654545,0.000000,0.000000,0.000000,6.971418,0.000000,6.229160,4,0,0,0,1


In [54]:
mean_expression = subset.to_df().mean(axis=0)

In [55]:
gene_sums = subset.X.sum(axis=0).A1  # Using .A1 to flatten the sparse matrix sum
gene_names = subset.var_names

In [56]:
gene_df = pd.DataFrame({'gene': gene_names, 'sum_expression': gene_sums})

In [57]:
sorted_gene_df = gene_df.sort_values(by='sum_expression', ascending=False)


In [58]:
top_genes = sorted_gene_df.head(1000)


In [59]:
top_genes.to_csv('validation/diff_gene/tumor_top_1000_genes.csv', index=False)


In [36]:
immune_cell_highly_expressed = pd.read_csv('validation/diff_gene/highly_expressed_immune_cell.csv')
t_cell_highly_expressed = pd.read_csv('validation/diff_gene/all_data_diff_gene_markers_top3000.csv')

In [37]:
immune_cell_highly_expressed.drop_duplicates(subset='gene', inplace=True)

In [38]:
immune_cell_highly_expressed

,gene
0,OLFML3
1,TNFSF4
2,PLXNA2
3,HECW2
4,PCDH12
...,...
160,IFI6
161,ERBB2
162,KRT7
163,CEACAM6


In [39]:
t_cell_highly_expressed = t_cell_highly_expressed[~t_cell_highly_expressed['gene'].isin(immune_cell_highly_expressed['gene'])]

In [40]:
t_cell_highly_expressed

,gene,logfoldchange,pvals,adjusted_pvals,average_count,Unnamed: 5
3,CD24,0.889778,7.230000e-142,1.310000e-137,1.541697,0.889778
4,MX1,0.790000,2.140000e-48,3.230000e-45,0.267711,0.790000
5,SPP1,-0.781636,1.060000e-25,6.200000e-23,0.160324,0.781636
7,LTF,0.663780,1.180000e-48,1.940000e-45,0.481263,0.663780
8,FAM118A,-0.663108,2.140000e-16,5.780000e-14,0.108173,0.663108
...,...,...,...,...,...,...
2995,RERE,-0.000107,9.985742e-01,1.000000e+00,0.206369,0.000107
2996,CTSA,-0.000085,9.985001e-01,1.000000e+00,0.406757,0.000085
2997,CYTH2,-0.000076,9.990819e-01,1.000000e+00,0.163470,0.000076
2998,YIPF5,-0.000035,9.996513e-01,1.000000e+00,0.107927,0.000035


In [41]:
t_cell_highly_expressed.to_csv('validation/diff_gene/all_data_diff_gene_markers_top4000_filtered.csv', index=False)